# Setting up Mlflow

In [28]:
# Install the following librairies (it is better to create a venv (or conda) virtual environment first and install these librairies in it)
%pip install mlflow       
%pip install --upgrade jinja2
%pip install --upgrade Flask
%pip install  setuptools

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [29]:
import mlflow

In [36]:
# ... (ton code avec PORT = 8080)
print(f"MLflow tracking URI ready: {TRACKING_URI}")

MLflow tracking URI ready: http://127.0.0.1:8000


In [35]:
# Start MLflow server automatically if it is not already running on 127.0.0.1:8080.
import atexit
import socket
import subprocess
import sys
import time
from pathlib import Path
HOST = "127.0.0.1"
PORT = 8000
TRACKING_URI = f"http://{HOST}:{PORT}"
def _port_open(host: str, port: int, timeout: float = 0.5) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0
def ensure_mlflow_server():
    if _port_open(HOST, PORT):
        return None
    db_path = Path("mlflow.db").resolve().as_posix()
    artifact_uri = Path("mlruns").resolve().as_uri()
    cmd = [
        sys.executable,
        "-m",
        "mlflow",
        "server",
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--backend-store-uri",
        f"sqlite:///{db_path}",
        "--default-artifact-root",
        artifact_uri,
    ]
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        creationflags=getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0),
    )
    for _ in range(40):
        if _port_open(HOST, PORT):
            return proc
        time.sleep(0.25)
    raise RuntimeError("MLflow server did not start on import mlflow")
mlflow.set_tracking_uri("http://127.0.0.1:8000")
_mlflow_server_process = ensure_mlflow_server()
if _mlflow_server_process is not None:
    atexit.register(lambda: _mlflow_server_process.terminate() if _mlflow_server_process.poll() is None else None)
print(f"MLflow tracking URI ready: {TRACKING_URI}")


MLflow tracking URI ready: http://127.0.0.1:8000


## Using the MLflow Client API


- Initiate a new Experiment.

- Start Runs within an Experiment.

- Document parameters, metrics, and tags for your Runs.

- Log artifacts linked to runs, such as models, tables, plots, and more.



In [37]:
from mlflow import MlflowClient
from pprint import pprint
from sklearn.ensemble import RandomForestRegressor


In [40]:
# Connect to the tracking server with an explicit client.
client = MlflowClient(tracking_uri=TRACKING_URI)
# It allows programmatic interaction with the MLflow tracking server.


We now have a client interface to the tracking server that can both send data to and retrieve data from the tracking server.



In [41]:
all_experiments = client.search_experiments()

print(all_experiments)


[<Experiment: artifact_location='/Users/baibrahima/Desktop/MLOPS-RETRIES/mlruns/0', creation_time=1773649534856, experiment_id='0', last_update_time=1773649534856, lifecycle_stage='active', name='Default', tags={}>]


### create an experiment

In [42]:
# Provide an Experiment description that will appear in the UI
experiment_description = (
    "This is the grocery forecasting project. "
    "This experiment contains the produce models for apples."
)

# Provide searchable tags that define characteristics of the Runs that
# will be in this Experiment
experiment_tags = {
    "project_name": "grocery-forecasting",
    "store_dept": "produce",
    "team": "stores-ml",
    "project_quarter": "Q3-2023",
    "mlflow.note.content": experiment_description,
}

# Get or create the Experiment to avoid RESOURCE_ALREADY_EXISTS
exp_name = "Apple_Models_2"
exp = client.get_experiment_by_name(exp_name)

if exp is None:
    experiment_id = client.create_experiment(name=exp_name, tags=experiment_tags)
else:
    experiment_id = exp.experiment_id

print("experiment_id:", experiment_id)


experiment_id: 1


In [43]:
# Use search_experiments() to search on the project_name tag key

apples_experiment = client.search_experiments(
    filter_string="tags.`project_name` = 'grocery-forecasting'"
)

print(vars(apples_experiment[0]))


{'_experiment_id': '1', '_name': 'Apple_Models_2', '_artifact_location': '/Users/baibrahima/Desktop/MLOPS-RETRIES/mlruns/1', '_lifecycle_stage': 'active', '_tags': {'project_name': 'grocery-forecasting', 'store_dept': 'produce', 'team': 'stores-ml', 'project_quarter': 'Q3-2023', 'mlflow.note.content': 'This is the grocery forecasting project. This experiment contains the produce models for apples.'}, '_creation_time': 1773651284094, '_last_update_time': 1773651284094}


### Create a dataset about apples

In [44]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


def generate_apple_sales_data_with_promo_adjustment(
    base_demand: int = 1000, n_rows: int = 5000
):
    """
    Generates a synthetic dataset for predicting apple sales demand with seasonality
    and inflation.

    This function creates a pandas DataFrame with features relevant to apple sales.
    The features include date, average_temperature, rainfall, weekend flag, holiday flag,
    promotional flag, price_per_kg, and the previous day's demand. The target variable,
    'demand', is generated based on a combination of these features with some added noise.

    Args:
        base_demand (int, optional): Base demand for apples. Defaults to 1000.
        n_rows (int, optional): Number of rows (days) of data to generate. Defaults to 5000.

    Returns:
        pd.DataFrame: DataFrame with features and target variable for apple sales prediction.

    Example:
        >>> df = generate_apple_sales_data_with_seasonality(base_demand=1200, n_rows=6000)
        >>> df.head()
    """

    # Set seed for reproducibility
    np.random.seed(9999)

    # Create date range
    dates = [datetime.now() - timedelta(days=i) for i in range(n_rows)]
    dates.reverse()

    # Generate features
    df = pd.DataFrame(
        {
            "date": dates,
            "average_temperature": np.random.uniform(10, 35, n_rows),
            "rainfall": np.random.exponential(5, n_rows),
            "weekend": [(date.weekday() >= 5) * 1 for date in dates],
            "holiday": np.random.choice([0, 1], n_rows, p=[0.97, 0.03]),
            "price_per_kg": np.random.uniform(0.5, 3, n_rows),
            "month": [date.month for date in dates],
        }
    )

    # Introduce inflation over time (years)
    df["inflation_multiplier"] = (
        1 + (df["date"].dt.year - df["date"].dt.year.min()) * 0.03
    )

    # Incorporate seasonality due to apple harvests
    df["harvest_effect"] = np.sin(2 * np.pi * (df["month"] - 3) / 12) + np.sin(
        2 * np.pi * (df["month"] - 9) / 12
    )

    # Modify the price_per_kg based on harvest effect
    df["price_per_kg"] = df["price_per_kg"] - df["harvest_effect"] * 0.5

    # Adjust promo periods to coincide with periods lagging peak harvest by 1 month
    peak_months = [4, 10]  # months following the peak availability
    df["promo"] = np.where(
        df["month"].isin(peak_months),
        1,
        np.random.choice([0, 1], n_rows, p=[0.85, 0.15]),
    )

    # Generate target variable based on features
    base_price_effect = -df["price_per_kg"] * 50
    seasonality_effect = df["harvest_effect"] * 50
    promo_effect = df["promo"] * 200

    df["demand"] = (
        base_demand
        + base_price_effect
        + seasonality_effect
        + promo_effect
        + df["weekend"] * 300
        + np.random.normal(0, 50, n_rows)
    ) * df[
        "inflation_multiplier"
    ]  # adding random noise

    # Add previous day's demand
    df["previous_days_demand"] = df["demand"].shift(1)
    df["previous_days_demand"].fillna(
        method="bfill", inplace=True
    )  # fill the first row

    # Drop temporary columns
    df.drop(columns=["inflation_multiplier", "harvest_effect", "month"], inplace=True)

    return df


In [45]:
data = generate_apple_sales_data_with_promo_adjustment()
data.head()

/var/folders/m3/r7vktzhx04gfw04ldx4tnc4w0000gn/T/ipykernel_33878/2753523027.py:89: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["previous_days_demand"].fillna(
/var/folders/m3/r7vktzhx04gfw04ldx4tnc4w0000gn/T/ipykernel_33878/2753523027.py:89: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["previous_days_demand"].fillna(


,date,average_temperature,rainfall,weekend,holiday,price_per_kg,promo,demand,previous_days_demand
0,2012-07-08 09:55:10.041775,30.584727,1.199291,1,0,1.726258,0,1151.276659,1151.276659
1,2012-07-09 09:55:10.041774,15.465069,1.037626,0,0,0.576471,0,906.836626,1151.276659
2,2012-07-10 09:55:10.041773,10.786525,5.656089,0,0,2.513328,0,857.895424,906.836626
3,2012-07-11 09:55:10.041772,23.648154,12.030937,0,0,1.839225,0,848.961007,857.895424
4,2012-07-12 09:55:10.041771,13.861391,4.303812,0,0,1.531772,0,983.128282,848.961007


### Logging our first runs with MLflow

In [46]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [47]:
# Set the global tracking URI for this session.
mlflow.set_tracking_uri(TRACKING_URI)


In [48]:
# Sets the current active experiment to the "Apple_Models" experiment and
# returns the Experiment metadata
apple_experiment = mlflow.set_experiment("Apple_Models_2")

# Define a run name for this iteration of training.
# If this is not set, a unique name will be auto-generated for your run.
run_name = "apples_rf_test_1"

# Define an artifact path that the model will be saved to.
artifact_path = "rf_apples_1"


In [49]:
# Split the data into features and target and drop irrelevant date field and target field
X = data.drop(columns=["date", "demand"])
y = data["demand"]

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

params = {
    "n_estimators": 12,
    "max_depth": 6,
    "min_samples_split": 8,
    "min_samples_leaf": 4,
    "bootstrap": True,
    "oob_score": False,
    "random_state": 888,
}

# Train the RandomForestRegressor
rf = RandomForestRegressor(**params)

# Fit the model on the training data
rf.fit(X_train, y_train)

# Predict on the validation set
y_pred = rf.predict(X_val)

# Calculate error metrics
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

# Assemble the metrics we're going to write into a collection
metrics = {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2}

# Initiate the MLflow run context
with mlflow.start_run(run_name=run_name) as run:
    # Log the parameters used for the model fit
    mlflow.log_params(params)

    # Log the error metrics that were calculated during validation
    mlflow.log_metrics(metrics)

    # Log an instance of the trained model for later use
    mlflow.sklearn.log_model(
        sk_model=rf, input_example=X_val, artifact_path=artifact_path
    )


2026/03/16 09:55:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/baibrahima/Desktop/MLOPS-RETRIES/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run apples_rf_test_1 at: http://127.0.0.1:8000/#/experiments/1/runs/1a3cc94a740e4f8dab17c8fd78eb2e1a
🧪 View experiment at: http://127.0.0.1:8000/#/experiments/1
